In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# 导入必要的基础类
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputRegressor

# 导入各种回归器
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn import metrics

# 导入其他回归模型
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.neighbors import KNeighborsRegressor
import torch
from sklearn.linear_model import LinearRegression
from tabpfn import TabPFNRegressor  # 添加TabPFN导入
from pytorch_tabnet.tab_model import TabNetRegressor
# 检查是否有 GPU 可用
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")
# 创建TabNet的包装器，使其符合scikit-learn的接口
# 定义改进后的多输出堆叠回归器
# TabTransformer包装器类定义
class TabTransformerWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, input_dim=None, output_dim=1, embedding_dim=72, depth=3, heads=1, 
                 attn_dropout=0.3864, ff_dropout=0.0, lr=0.0006774, 
                 batch_size=65, epochs=100, device='auto', random_state=42):
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.embedding_dim = embedding_dim
        self.depth = depth
        self.heads = heads
        self.attn_dropout = attn_dropout
        self.ff_dropout = ff_dropout
        self.lr = lr
        self.batch_size = batch_size
        self.epochs = epochs
        self.device = device
        self.random_state = random_state
        self.model = None
        
    def fit(self, X, y):
        import torch
        import torch.nn as nn
        from torch.utils.data import Dataset, DataLoader
        import numpy as np
        
        # 设置随机种子
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)
        
        # 自动确定输入维度
        if self.input_dim is None:
            self.input_dim = X.shape[1]
            
        # 处理目标变量维度
        if len(y.shape) == 1:
            y = y.reshape(-1, 1)
            self.output_dim = 1
        else:
            self.output_dim = y.shape[1]
            
        # 确定设备
        if self.device == 'auto':
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        
        # 数据集类
        class TabularDataset(Dataset):
            def __init__(self, X, y):
                self.X = torch.tensor(X, dtype=torch.float32)
                self.y = torch.tensor(y, dtype=torch.float32)
                
            def __len__(self):
                return len(self.X)
            
            def __getitem__(self, idx):
                return self.X[idx], self.y[idx]
        
        # Transformer模型定义
        class TabTransformer(nn.Module):
            def __init__(self, input_dim, output_dim, embedding_dim, depth, heads, attn_dropout, ff_dropout):
                super().__init__()
                self.input_dim = input_dim
                self.output_dim = output_dim
                
                # 嵌入层
                self.embedding = nn.Linear(input_dim, embedding_dim)
                
                # 位置编码
                self.pos_encoder = nn.Parameter(torch.zeros(1, input_dim, embedding_dim))
                
                # Transformer编码器层
                encoder_layers = nn.TransformerEncoderLayer(
                    d_model=embedding_dim,
                    nhead=heads,
                    dim_feedforward=embedding_dim * 4,
                    dropout=attn_dropout,
                    activation='gelu',
                    batch_first=True
                )
                self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=depth)
                
                # MLP头部
                self.mlp_head = nn.Sequential(
                    nn.Linear(embedding_dim * input_dim, embedding_dim * 2),
                    nn.LayerNorm(embedding_dim * 2),
                    nn.GELU(),
                    nn.Dropout(ff_dropout),
                    nn.Linear(embedding_dim * 2, embedding_dim),
                    nn.LayerNorm(embedding_dim),
                    nn.GELU(),
                    nn.Dropout(ff_dropout),
                    nn.Linear(embedding_dim, output_dim)
                )
                
            def forward(self, x):
                batch_size = x.size(0)
                
                # 重塑输入
                x = x.unsqueeze(-1)
                
                # 为每个特征创建嵌入
                x = x.repeat(1, 1, self.embedding.in_features)
                x = x * torch.eye(self.embedding.in_features, device=x.device)
                
                # 应用线性嵌入
                x = self.embedding(x)  # shape: [batch_size, num_features, embedding_dim]
                
                # 添加位置编码
                x = x + self.pos_encoder
                
                # 通过Transformer编码器
                x = self.transformer_encoder(x)
                
                # 重塑输出
                x = x.reshape(batch_size, -1)
                
                # 通过MLP头部
                x = self.mlp_head(x)
                
                return x
        
        # 创建模型
        self.model = TabTransformer(
            input_dim=self.input_dim,
            output_dim=self.output_dim,
            embedding_dim=self.embedding_dim,
            depth=self.depth,
            heads=self.heads,
            attn_dropout=self.attn_dropout,
            ff_dropout=self.ff_dropout
        ).to(self.device)
        
        # 创建数据集和数据加载器
        train_dataset = TabularDataset(X, y)
        train_loader = DataLoader(
            train_dataset, 
            batch_size=self.batch_size, 
            shuffle=True
        )
        
        # 定义优化器和损失函数
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        loss_fn = nn.MSELoss()
        
        # 训练模型，使用tqdm显示进度条
        self.model.train()
        for epoch in range(self.epochs):
            epoch_loss = 0
            for x_batch, y_batch in train_loader:
                x_batch = x_batch.to(self.device)
                y_batch = y_batch.to(self.device)
                
                # 前向传播
                optimizer.zero_grad()
                y_pred = self.model(x_batch)
                loss = loss_fn(y_pred, y_batch)
                
                # 反向传播
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
        
        return self
    
    def predict(self, X):
        import torch
        import numpy as np
        
        if self.model is None:
            raise ValueError("Model not fitted yet.")
        
        # 准备数据
        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        
        # 分批预测以避免内存问题
        batch_size = 128
        predictions = []
        
        self.model.eval()
        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                batch = X_tensor[i:i+batch_size]
                batch_preds = self.model(batch).cpu().numpy()
                predictions.append(batch_preds)
                
        predictions = np.vstack(predictions)
        
        # 如果输出是单变量，返回一维数组
        if self.output_dim == 1:
            return predictions.ravel()
        return predictions
# TabNet包装器类，使其符合scikit-learn的接口
class TabNetRegressorWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, n_d=22, n_a=128, n_steps=3, gamma=2.0, lambda_sparse=0.0005467072969055289, 
                 optimizer_fn=torch.optim.Adam, optimizer_params=dict(lr=0.029384293039604333), 
                 scheduler_params={"step_size":50, "gamma":0.8}, scheduler_fn=torch.optim.lr_scheduler.StepLR, 
                 mask_type='entmax', batch_size=227, virtual_batch_size=96, max_epochs=200, 
                 patience=20, device_name='auto', verbose=0, seed=42):
        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.gamma = gamma
        self.lambda_sparse = lambda_sparse
        self.optimizer_fn = optimizer_fn
        self.optimizer_params = optimizer_params
        self.scheduler_params = scheduler_params
        self.scheduler_fn = scheduler_fn
        self.mask_type = mask_type
        self.batch_size = batch_size
        self.virtual_batch_size = virtual_batch_size
        self.max_epochs = max_epochs
        self.patience = patience
        self.device_name = device_name
        self.verbose = verbose
        self.seed = seed
        self.model = None
        
    def fit(self, X, y):
        if len(y.shape) == 1:
            y = y.reshape(-1, 1)
            
        # 创建模型
        self.model = TabNetRegressor(
            n_d=self.n_d,
            n_a=self.n_a,
            n_steps=self.n_steps,
            gamma=self.gamma,
            lambda_sparse=self.lambda_sparse,
            optimizer_fn=self.optimizer_fn,
            optimizer_params=self.optimizer_params,
            scheduler_params=self.scheduler_params,
            scheduler_fn=self.scheduler_fn,
            mask_type=self.mask_type,
            device_name=self.device_name,
            verbose=self.verbose,
            seed=self.seed
        )
        
        # 抑制输出
        import contextlib
        import io
        with contextlib.redirect_stdout(io.StringIO()):
            # 训练模型
            self.model.fit(
                X_train=X,
                y_train=y,
                max_epochs=self.max_epochs,
                patience=self.patience,
                batch_size=self.batch_size,
                virtual_batch_size=self.virtual_batch_size,
                num_workers=0
            )
        return self
    
    def predict(self, X):
        if self.model is None:
            raise ValueError("Model not fitted yet.")
        preds = self.model.predict(X)
        if preds.shape[1] == 1:
            return preds.ravel()
        return preds


from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.model_selection import KFold
import numpy as np

from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import make_pipeline

def make_fold_estimator(estimator):
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), clone(estimator)),
        transformer=StandardScaler()
    )


class MultiOutputStackingRegressor(BaseEstimator, RegressorMixin):
    """
    多输出堆叠回归器，支持交叉验证
    
    Parameters
    ----------
    estimators : list
        包含(名称, 估计器)元组的列表，构成第一级学习器
    final_estimator : estimator
        用于组合第一级预测的最终估计器
    cv : int, cross-validation generator or iterable, default=5
        确定交叉验证拆分策略
    """
    def __init__(self, estimators, final_estimator, cv=5):
        self.estimators = estimators
        self.final_estimator = final_estimator
        self.cv = cv
        self.base_models = {}  # 存储基础模型
        self.meta_models = {}  # 存储元模型
    
    def fit(self, X, Y):
        n_targets = Y.shape[1]
        n_samples = X.shape[0]
        n_estimators = len(self.estimators)
        
        # 初始化存储交叉验证的元特征
        meta_features = np.zeros((n_samples, n_estimators, n_targets))
        
        # 为每个目标创建模型存储
        for target_idx in range(n_targets):
            self.base_models[target_idx] = []
        
        # 第一阶段：使用交叉验证训练基础模型
        kf = KFold(n_splits=self.cv, shuffle=True, random_state=42)
        
        # 对每个目标分别进行基础模型训练
        for target_idx in range(n_targets):
            y_target = Y[:, target_idx]
            
            # 对每个基础学习器进行训练
            for est_idx, (name, estimator) in enumerate(self.estimators):
                print(f"训练目标 {target_idx+1}/{n_targets} 的 {name} 模型...")
                
                # 使用交叉验证生成元特征
                for train_idx, val_idx in kf.split(X):
                    X_train, X_val = X[train_idx], X[val_idx]
                    y_train = y_target[train_idx]
                    
                    # 训练模型
                    model = make_fold_estimator(estimator).fit(X_train, y_train)
                    
                    # 生成验证集的预测作为元特征
                    meta_features[val_idx, est_idx, target_idx] = model.predict(X_val)
                
                # 在全部数据上训练最终模型
                final_model = make_fold_estimator(estimator).fit(X, y_target)
                self.base_models[target_idx].append(final_model)
        
        # 第二阶段：训练元模型
        for target_idx in range(n_targets):
            print(f"训练目标 {target_idx+1}/{n_targets} 的元模型...")
            
            # 提取当前目标变量的元特征
            target_meta_features = meta_features[:, :, target_idx]
            y_target = Y[:, target_idx]
            
            # 训练元模型
            meta_model = clone(self.final_estimator)
            meta_model.fit(target_meta_features, y_target)
            self.meta_models[target_idx] = meta_model
        
        return self
    
    def predict(self, X):
        n_samples = X.shape[0]
        n_targets = len(self.meta_models)
        n_estimators = len(self.estimators)
        
        # 存储基础模型预测
        meta_features = np.zeros((n_samples, n_estimators, n_targets))
        
        # 对每个目标变量，使用所有基础模型进行预测
        for target_idx in range(n_targets):
            for est_idx, model in enumerate(self.base_models[target_idx]):
                meta_features[:, est_idx, target_idx] = model.predict(X)
        
        # 元模型预测最终结果
        y_pred = np.zeros((n_samples, n_targets))
        for target_idx, meta_model in self.meta_models.items():
            meta_features_target = meta_features[:, :, target_idx]
            y_pred[:, target_idx] = meta_model.predict(meta_features_target)
        
        return y_pred
    
    def get_meta_features(self, X, target_idx=0):
        """获取指定目标的元特征"""
        if target_idx not in self.meta_models:
            raise ValueError(f"Target index {target_idx} not found in trained models")
            
        n_samples = X.shape[0]
        n_estimators = len(self.estimators)
        
        # 存储指定目标的基础模型预测
        meta_features = np.zeros((n_samples, n_estimators))
        
        # 获取每个基础模型对指定目标的预测
        for est_idx, model in enumerate(self.base_models[target_idx]):
            meta_features[:, est_idx] = model.predict(X)
            
        return meta_features
    
    def get_meta_model(self, target_idx=0):
        """获取指定目标的元模型"""
        if target_idx not in self.meta_models:
            raise ValueError(f"Target index {target_idx} not found in trained models")
            
        return self.meta_models[target_idx]
    
    def get_base_models(self, target_idx=0):
        """获取指定目标的所有基础学习器"""
        if target_idx not in self.base_models:
            raise ValueError(f"Target index {target_idx} not found in trained models")
            
        return self.base_models[target_idx]

# 导入原始数据
df_original = pd.read_excel('data/pinghua_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 提取目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 从原始数据中分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 将原始数据分为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_original, y_original, test_size=0.3, random_state=42
)

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 创建标准化器
# Fold-local preprocessing is handled by the base-estimator pipelines.
X_train_array = X_train.to_numpy(dtype=float)
y_train_array = y_train.to_numpy(dtype=float)
X_test_array = X_test.to_numpy(dtype=float)
y_test_array = y_test.to_numpy(dtype=float)

base_learners = [
     ("RF", RandomForestRegressor(
        n_estimators=200,  # 减少估计器数量以提高速度
        max_depth=20,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",  # 使用sqrt特征以加速
        n_jobs=-1,
        random_state=42
    )),
    ("ET",ExtraTreesRegressor(
    n_estimators=500,    
    max_depth=30,        
    min_samples_split=2, 
    min_samples_leaf=1,  
    max_features=None,   
    bootstrap=False,     
    n_jobs=-1,           
    random_state=42
)),
("KNN",KNeighborsRegressor(
    n_neighbors=2,  # 邻居数量，可以进行调整
    weights='distance',  # 权重类型：'uniform'或'distance'
    algorithm='auto',  # 算法：'auto', 'ball_tree', 'kd_tree', 'brute'
    p=1,  # 距离度量，2表示欧氏距离
    n_jobs=-1  # 使用所有可用的CPU核心
)),("LGBM", LGBMRegressor(
        boosting_type= 'gbdt',
        objective= 'regression',
        metric= 'rmse',
        num_leaves= 64,
        max_depth= 14,
        learning_rate= 0.10279735193954564,
        feature_fraction= 0.792740064845325,
        subsample= 0.7533616827994455,
        min_child_samples= 17,
        subsample_freq= 5,
        verbose= -1,
        n_jobs= -1,
        n_estimators= 335,
        random_state= 42
    )),
 ("TabNet", TabNetRegressorWrapper(
        n_d=22,                      # 决策层维度
        n_a=128,                     # 注意力层维度
        n_steps=3,                   # 步数
        gamma=2.0,                   # 稀疏度
        lambda_sparse=0.0005467072969055289,  # 额外的稀疏度
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=0.029384293039604333),  # 学习率
        scheduler_params={"step_size":50, "gamma":0.8},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax',
        device_name=device,
        batch_size=227,
        virtual_batch_size=96,
        max_epochs=100,  # 减少训练轮数以加快训练速度
        patience=20,
        verbose=0,
        seed=42
    )),
    ("TabPFN", TabPFNRegressor(device=device,)),
    ("TabTransformer", TabTransformerWrapper(
        embedding_dim=72,
        depth=3,
        heads=1,
        attn_dropout=0.3864,
        ff_dropout=0.0,
        lr=0.0006774,
        batch_size=65,
        epochs=50,  # 减少训练轮数以加快速度
        device='auto',
        random_state=42
    ))  ,
    ("XGB", XGBRegressor(
        n_estimators=500,
        max_depth=15,
        learning_rate=0.01,
        subsample=0.5,
        colsample_bytree=1.0,
        gamma=0.0,
        reg_alpha=0.0,
        reg_lambda=0.0,
        n_jobs=-1,
        random_state=42
    )),("Ridge", Ridge(
    alpha=0.9609812947036424,              # 降低正则化强度
    fit_intercept=True,
    copy_X=True,
    max_iter=947,          # 增加最大迭代次数
    tol=0.001562069367563987,             # 提高精度
    random_state=42,
    solver='sag'
    )),
]

# 定义二级学习器
meta_model = Ridge(alpha=1.0, random_state=42)
# meta_model = LinearRegression()

# 创建改进的多输出堆叠回归器，使用5折交叉验证
stacking_regressor = MultiOutputStackingRegressor(
    estimators=base_learners, 
    final_estimator=meta_model,
    cv=5
)

# 进行交叉验证评估
print("\n开始进行交叉验证评估...")
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

cv_scores = {var: [] for var in target_vars}
cv_rmse = {var: [] for var in target_vars}
cv_mae = {var: [] for var in target_vars}

for train_idx, val_idx in kf.split(X_train_array):
    # 分割训练集和验证集
    X_train_fold, X_val_fold = X_train_array[train_idx], X_train_array[val_idx]
    y_train_fold, y_val_fold = y_train_array[train_idx], y_train_array[val_idx]
    
    # 训练堆叠模型
    stacking_regressor.fit(X_train_fold, y_train_fold)
    
    # 在验证集上预测
    y_val_pred = stacking_regressor.predict(X_val_fold)
    
    # 计算每个目标变量的指标
    for i, var in enumerate(target_vars):
        r2 = metrics.r2_score(y_val_fold[:, i], y_val_pred[:, i])
        rmse = np.sqrt(metrics.mean_squared_error(y_val_fold[:, i], y_val_pred[:, i]))
        mae = metrics.mean_absolute_error(y_val_fold[:, i], y_val_pred[:, i])
        
        cv_scores[var].append(r2)
        cv_rmse[var].append(rmse)
        cv_mae[var].append(mae)

# 打印交叉验证结果
print(f"\n{k_folds}折交叉验证结果:")
mean_cv_r2 = 0
mean_cv_rmse = 0
mean_cv_mae = 0

for var in target_vars:
    mean_r2 = np.mean(cv_scores[var])
    mean_rmse = np.mean(cv_rmse[var])
    mean_mae = np.mean(cv_mae[var])
    
    mean_cv_r2 += mean_r2
    mean_cv_rmse += mean_rmse
    mean_cv_mae += mean_mae
    
    print(f"\n{var} 交叉验证指标:")
    print(f"平均 R²: {mean_r2:.4f} ± {np.std(cv_scores[var]):.4f}")
    print(f"平均 RMSE: {mean_rmse:.4f} ± {np.std(cv_rmse[var]):.4f}")
    print(f"平均 MAE: {mean_mae:.4f} ± {np.std(cv_mae[var]):.4f}")

# 计算所有目标变量的平均指标
mean_cv_r2 /= len(target_vars)
mean_cv_rmse /= len(target_vars)
mean_cv_mae /= len(target_vars)

print("\n所有变量平均交叉验证指标:")
print(f"平均 R²: {mean_cv_r2:.4f}")
print(f"平均 RMSE: {mean_cv_rmse:.4f}")
print(f"平均 MAE: {mean_cv_mae:.4f}")

# 在完整训练集上训练最终堆叠模型
print("\n开始训练最终堆叠模型...")
stacking_regressor.fit(X_train_array, y_train_array)
print("堆叠模型训练完成!")

# 使用训练好的模型在测试集上预测
y_pred_test_array = stacking_regressor.predict(X_test_array)

# 将预测结果反标准化回原始空间
y_pred_test = y_pred_test_array

# 真实值
y_test_true = y_test.values

# 为每个输出目标计算评估指标
print("\n堆叠模型测试集评估指标:")
overall_r2 = 0
overall_rmse = 0
overall_mae = 0

for idx, name in enumerate(target_vars):
    # 测试集指标
    mse_test = metrics.mean_squared_error(y_test_true[:, idx], y_pred_test[:, idx])
    rmse_test = np.sqrt(mse_test)
    mae_test = metrics.mean_absolute_error(y_test_true[:, idx], y_pred_test[:, idx])
    r2_test = metrics.r2_score(y_test_true[:, idx], y_pred_test[:, idx])
    
    overall_r2 += r2_test
    overall_rmse += rmse_test
    overall_mae += mae_test
    
    print(f"\n{name} 指标:")
    print(f"MSE={mse_test:.4f}, RMSE={rmse_test:.4f}, MAE={mae_test:.4f}, R²={r2_test:.4f}")
    print('-----------------------------------')

# 计算所有目标的平均性能指标
overall_r2 /= len(target_vars)
overall_rmse /= len(target_vars)
overall_mae /= len(target_vars)

print("\n总体平均指标:")
print(f"平均 RMSE: {overall_rmse:.4f}")
print(f"平均 MAE: {overall_mae:.4f}")
print(f"平均 R²: {overall_r2:.4f}")

# 与原始模型比较
print("\n与原始无交叉验证模型的性能比较:")
print("改进后的模型(使用交叉验证)平均 R²: {:.4f}".format(overall_r2))
print("原始模型(无交叉验证)平均 R²: 0.6958")


In [ ]:
import seaborn as sns
# 添加到现有代码结尾处，SHAP分析完成后
import matplotlib.pyplot as plt
# 创建一个函数来标准化SHAP值
def normalize_shap(shap_values):
    """将SHAP值标准化到[-1, 1]范围内"""
    abs_max = np.abs(shap_values).max()
    if abs_max > 0:
        return shap_values / abs_max
    return shap_values

# 存储所有目标的元模型SHAP值
all_meta_shap_values = []
all_meta_features = []
base_model_names = [name for name, _ in base_learners]

# 为每个目标变量计算并收集SHAP值
print("\n计算所有目标变量的归一化SHAP值汇总...")
for target_idx, target_name in enumerate(target_vars):
    try:
        # 获取这个目标的元模型
        meta_model = stacking_regressor.get_meta_model(target_idx)
        
        # 获取一级学习器的预测结果
        meta_features = stacking_regressor.get_meta_features(X_test_array, target_idx)
        all_meta_features.append(meta_features)
        
        # 使用KernelExplainer分析元模型
        meta_explainer = shap.KernelExplainer(meta_model.predict, meta_features)
        meta_shap_values = meta_explainer.shap_values(meta_features)
        
        # 标准化SHAP值
        normalized_shap = normalize_shap(meta_shap_values)
        all_meta_shap_values.append(normalized_shap)
        
        print(f"  已处理目标 {target_name} 的SHAP值")
        
    except Exception as e:
        print(f"  处理目标 {target_name} 时出错: {str(e)}")

# 确保我们至少有一些SHAP值可用
if all_meta_shap_values:
    # 计算归一化后的平均SHAP值
    avg_meta_features = np.mean(all_meta_features, axis=0)
    avg_meta_shap = np.mean(all_meta_shap_values, axis=0)
    
    # 创建汇总可视化
    plt.figure(figsize=(12, 8), dpi=300)
    
    # 点图 - 显示所有基础学习器的综合贡献
    shap.summary_plot(
        avg_meta_shap,
        avg_meta_features,
        feature_names=base_model_names,
        plot_type="dot",
        show=False,
        cmap=plt.cm.Purples_r  # 使用蓝绿色调
    )
    plt.title("Overall Meta Learner - Base Learner Contribution Analysis (All Target Variables)", fontsize=16)
    plt.tight_layout()
    plt.show()

print("\nNormalized SHAP value analysis completed!")


In [ ]:
# 检查当前 base_model_names 中是否包含 'TabTransformer'
if 'TabTransformer' in base_model_names:
    # 找到 'TabTransformer' 的索引
    index = base_model_names.index('TabTransformer')
    # 将其替换为带有换行符的版本
    base_model_names[index] = 'TabTra-\nnsformer'

In [ ]:
import shap

# 假设 X_test 是一个 Pandas DataFrame
feature_importance = np.mean(np.abs(avg_meta_shap), axis=0)

# 创建一个 DataFrame，将列名设置为 X_test 的特征名
feature_importance_df = pd.DataFrame({
    'Feature': base_model_names,  # 使用 X_test 的特征名
    'Importance': feature_importance  # 对应的均值
}).sort_values(by='Importance', ascending=False)  # 按重要性降序排序
feature_importance_df

In [ ]:
import os
import numpy as np
import pandas as pd
import json

# 创建目录（如果不存在）
os.makedirs("plot/shap", exist_ok=True)

# 提取特征名和重要性值
features = feature_importance_df['Feature']
importances = feature_importance_df['Importance']

# 将 NumPy 数组和 Pandas Series 转换为列表，提高兼容性
def convert_to_serializable(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, pd.Series):
        return obj.tolist()
    elif isinstance(obj, pd.DataFrame):
        return obj.to_dict('records')
    elif isinstance(obj, list) and len(obj) > 0 and isinstance(obj[0], np.ndarray):
        return [x.tolist() for x in obj]
    elif hasattr(obj, 'tolist'):  # 处理其他可转换为列表的对象
        return obj.tolist()
    else:
        return obj

# 保存 DataFrame 到 CSV
feature_importance_df.to_csv("plot/shap/feature_importance.csv", index=False)

# 保存其他变量到 JSON 文件（更好的跨版本/环境兼容性）
variables_to_save = {
    'features': convert_to_serializable(features),
    'importances': convert_to_serializable(importances),
    'avg_meta_shap': convert_to_serializable(avg_meta_shap),
    'avg_meta_features': convert_to_serializable(avg_meta_features),
    'base_model_names': convert_to_serializable(base_model_names)
}

with open("plot/shap/plot_variables.json", 'w') as f:
    json.dump(variables_to_save, f)

print("变量已保存到 plot/shap 目录下的 CSV 和 JSON 文件中")